In [1]:
import pandas as pd
from random import random

In [2]:
roles_df = pd.read_csv('framebank.csv')

In [3]:
def pair_differences_not_greater(lst, max_diff):
    lst = sorted(lst)
    for i in range(1, len(lst)):
        if abs(lst[i-1] - lst[i]) > max_diff:
            return False
    return True

In [4]:
def simple_kwic_output(kwics):
    if isinstance(kwics[0], list):
        return "\n".join(' '.join([tk.form for tk in kwic]).replace(' ,', ',').replace(' .', '.') for kwic in kwics)
    elif isinstance(kwics, str):
        return kwics
    else:
        return ' '.join([tk.form for tk in kwics])

In [5]:
class Token:
    def __init__(self, token_info):
        self.id, self.form, self.lemma, self.pos, self.xpos, self.gr, self.head, self.deprel, self.deps, self.misc = token_info.split('\t')
        self.id = int(self.id)
        if self.gr != '_':
            for cath_value in self.gr.split('|'):
                cathegory, value = cath_value.split('=')
                setattr(self, cathegory.lower(), value)
            
    def __str__(self):
        return '\t'.join(f'{k}:{v}' for k, v in self.__dict__.items() if not k.startswith('__') and not callable(k))
    
    def __repr__(self):
        return '\t'.join(f'{k}:{v}' for k, v in self.__dict__.items() if not k.startswith('__') and not callable(k))

In [6]:
class Sentence:
    def __init__(self, sentence_info):
        self.sent_id, self.sentence_text, *self.tokens = sentence_info.split('\n')
        self.sentence_text = self.sentence_text.replace('# text = ', '')
        self.tokens = [Token(token_info=token) for token in self.tokens]
    
    def __str__(self):
        return f'{self.sent_id}\n{self.sentence_text}\n'+'\n'.join(str(token) for token in self.tokens)
    
    def __repr__(self):
        return f'{self.sent_id}\n{self.sentence_text}\n'+'\n'.join(str(token) for token in self.tokens)
    
    def _search_by_token(self, query_token:str, kwic_len:int):
        if query_token in self.sentence_text:
            for tk in self.tokens:
                if tk.form == query_token:
                    query_token_index = tk.id
                    kwik = [tk for tk in self.tokens if abs(tk.id-query_token_index) <= kwic_len]
                    return kwik
        
    def _search_by_lemma(self, query_lemma:str, kwic_len:int):
        for tk in self.tokens:
            if tk.lemma == query_lemma:
                query_token_index = tk.id
                kwik = [tk for tk in self.tokens if abs(tk.id-query_token_index) <= kwic_len]
                return kwik
                
    def _general_search(self, kwic_len:int, token=None, lemma=None, pos=None, xpos=None, gr=None, deprel=None, **kwargs):
        query = {'form': token, 'lemma':lemma, 'pos': pos, 'xpos': xpos, 'gr': gr, 'deprel': deprel}
        query = {k:v for k, v in query.items() if v is not None}
        if kwargs:
            for k, v in kwargs.items():
                query[k] = v
        
        for tk in self.tokens:
            condition = all(tk.__getattribute__(key)==value for key, value in query.items())
            if condition:
                query_token_index = tk.id
                kwik = [tk for tk in self.tokens if abs(tk.id-query_token_index) <= kwic_len]
                return kwik
    
    def _multiword_search(self, token_descriptions, kwik_len=1, allow_distance=1): # token_descriptions is list of dicts that describe each token with some features
        all_tokens_present = all(
            any(
                all(tk.__getattribute__(key)==value for key, value in tk_descripion.items()) 
                    for tk in self.tokens
                )
                for tk_descripion in token_descriptions
        )
        if all_tokens_present:
            token_indexes = []
            for token_description in token_descriptions:
                for tk in self.tokens:
                    condition = all(tk.__getattribute__(key)==value for key, value in token_description.items())
                    if condition:
                        token_indexes.append(tk.id)
            if pair_differences_not_greater(token_indexes, allow_distance):
                kwik = [tk for tk in self.tokens 
                        if (
                            abs(min(token_indexes) - tk.id) <= kwik_len 
                            or abs(max(token_indexes) - tk.id) <= kwik_len
                            or tk.id >=min(token_indexes) and tk.id <= max(token_indexes)
                            )
                        ]
                return kwik

In [7]:
class Corpus:
    def __init__(self):
        self.sentences = []

    def load_from_file(self, filepath):
        with open(filepath, encoding='utf-8') as corpus_file:
            self.sentences = [Sentence(sent) for sent in corpus_file.read().split('\n\n') if sent]

    def search_by_token(self, token, n_examples=5, kwic_len=5):
        qwery_answer = []
        for sentence in sorted(self.sentences, key=lambda x: random()):
            if len(qwery_answer) == n_examples:
                return qwery_answer
            c = sentence._search_by_token(query_token=token, kwic_len=kwic_len)
            if c:
                qwery_answer.append(c)
        if qwery_answer:
            return qwery_answer
        return f'Примеров для {token=} в корпусе не нашлось.'
    
    def search_by_lemma(self, lemma, n_examples=5, kwic_len=5):
        qwery_answer = []
        for sentence in sorted(self.sentences, key=lambda x: random()):
            if len(qwery_answer) == n_examples:
                return qwery_answer
            c = sentence._search_by_lemma(query_lemma=lemma, kwic_len=kwic_len)
            if c:
                qwery_answer.append(c)
        if qwery_answer:
            return qwery_answer
        return f'Примеров для {lemma=} в корпусе не нашлось.'

    def general_search(self, token=None, lemma=None, pos=None, xpos=None, gr=None, deprel=None, n_examples=5, kwic_len=5, **kwargs):
        qwery_answer = []
        for sentence in sorted(self.sentences, key=lambda x: random()):
            if len(qwery_answer) == n_examples:
                return qwery_answer
            c = sentence._general_search(token=token, lemma=lemma, pos=pos, xpos=xpos, gr=gr, deprel=deprel, kwic_len=kwic_len, **kwargs)
            if c:
                qwery_answer.append(c)
        if qwery_answer:
            return qwery_answer
        plug = ' '.join([f'{x=}' for x in [lemma, pos, xpos, gr, deprel] if x is not None])
        return f'Примеров для {plug} в корпусе не нашлось.'
    
    def search_multiword(self, token_descriptions, kwik_len=1, allow_distance=1, n_examples=5, ):
        qwery_answer = []
        for sentence in sorted(self.sentences, key=lambda x: random()):
            if len(qwery_answer) == n_examples:
                return qwery_answer
            c = sentence._multiword_search(token_descriptions, kwik_len, allow_distance)
            if c:
                qwery_answer.append(c)
        if qwery_answer:
            return qwery_answer
        return 'Поиск не дал результатов'
    
    def search_by_role(self, keyword_lemma='', role='', kwik_len=1, allow_distance=1, n_examples=5):
        if keyword_lemma and role:
            examples = roles_df[(roles_df.KeyLexemes==keyword_lemma) & (roles_df.Role==role)]
        elif keyword_lemma:
            examples = roles_df[roles_df.KeyLexemes==keyword_lemma]
        elif role:
            examples = roles_df[roles_df.Role==role]
        else:
            return 'Задан пустой запрос'
        output = [] 
        for example in examples.values:
            collocate = example[0]
            keyword_lemma = keyword_lemma if keyword_lemma else example[4]
            
            if collocate.isalpha():
                response = simple_kwic_output(self.search_multiword([{'lemma': keyword_lemma}, {'form': collocate}], kwik_len, allow_distance, n_examples))
                if response != 'Поиск не дал результатов':
                    output.append([example[2], collocate, response])
            
            elif all(l.isalpha() or l.isspace() for l in collocate):
                query = [{'lemma': keyword_lemma}] + [{'form': part} for part in collocate.split()]
                response = simple_kwic_output(self.search_multiword(query, kwik_len, allow_distance, n_examples))
                if response != 'Поиск не дал результатов':
                    output.append([example[2], collocate, response])
            
            else:
                print('WRONG TYPE COLLOCATE', collocate, 'LEMMA', keyword_lemma)
        return output

In [8]:
CORPUS = Corpus()
CORPUS.load_from_file('NPlus1.txt')

In [9]:
print(simple_kwic_output(CORPUS.general_search(lemma='видеть', kwic_len=10, n_examples=10)))

В рамках виртуального тура можно увидеть могилу Святого Петра, алтарную сень, 136-метровый центральный купол
спустя полторы минуты просили вспомнить, какие из них они видели. Во время показа изображений ученые вели запись сигналов,
Ученые смогли увидеть процессы, происходящие в первые 100 фемтосекунд ( одна десяти
Наиболее часто рядовые пользователи видят коды ошибок : 403 ( доступ запрещен ), 404
Crew Dragon на орбите Земли, также в кадре можно увидеть элементы скафандров экипажа. В мае 2015 года компания успешнопротестироваласистему аварийного
еще и такие английские слова, которых они никогда не видели ( с вероятностью, существенно превосходящей случайную ).
Снимки, полученные с нее, позволили увидеть детали таинственных пятен на Церере : в частности, стали
Тоесть если впервый раз птенцы видели два мячика, тововторой раз они следовали незамячиком ицилиндром,
, Германия ), -- " ATLASGALпредоставляет нам замечательную возможность увидеть, где именно образуется следующее поколение масси

In [10]:
print(simple_kwic_output(CORPUS.search_multiword([{'lemma': 'видеть'}, {'lemma': 'глаз'}], kwik_len=10, n_examples=10, allow_distance=2)))

располагают еще один светоделитель, пропускающий лишь половину фотонов. Напрямую увидеть глазом единичные фотоны невозможно -- светочувствительные клетки сетчатки требуютпо меньшей мере
оптического диапазона ( 400 - 800 нанометров ), наш глаз способен видеть и инфракрасное излучение ( около 1000 нанометров ), благодаря
В своих дальнейших экспериментах Холмс надеется " увидеть " глазами подопытных суперпозицию фотонов. Оптические возможности сетчатки до сих пор являются


In [11]:
print(simple_kwic_output(CORPUS.search_multiword([{'lemma': 'видеть'}, {'lemma': 'мочь'}], kwik_len=10, n_examples=10, allow_distance=1)))

только 536 людей из семи миллиардов побывали в космосе и смогли увидеть наш мир совсем по-другому, поэтому цель проекта -- дать
Стаким функционалом фармакологи смогут увидеть как быстро усваивается лекарственный препарат, физиологи - - отследить
Например, птицы не могут видеть, что происходит за спиной, а рыбы способны получать
" Мывпервые смогли увидеть излучение иатомарного водорода, имонооксида углерода вгалактике, которая находится
конкретного места. Как отмечают разработчики, пользуясь новой функцией пользователь сможет увидеть ежедневные маршруты и примечательные места, которые, возможно,
Отделив реакторный вклад, исследователи смогли увидеть в 2010 году примерно 10 геонейтрино.
смещают картинку на нашлемном дисплее. Благодаря системе кругового обзора летчик может увидеть, что происходит, например, под или за самолетом
Последний может видеть объекты на дальности до ста метров.
Так, относительная разница в длине плечей интерферометра, которую могла видеть Advanced LIGO,составила о

In [12]:
print(simple_kwic_output(CORPUS.search_multiword([{'lemma': 'видеть'}, {'pos': 'NOUN'}], kwik_len=10, n_examples=10, allow_distance=2)))

Благодаря этому боец увидит объемное изображение.
Тогда ученые увидели ватмосфере карликовой планеты присутствие небольшого количества водяного пара.
" Было изумительно увидеть эти особенности так отчетливо.
Врезультате они смогли увидеть вВеликом аттракторе 883галактики, треть которых не наблюдалась ранее.
Ввысоком разрешении ееможно увидеть здесь.
Обладатели подобной формы синестезии видят разные буквы и цифры разным цветом.
" Мывпервые смогли увидеть излучение иатомарного водорода, имонооксида углерода вгалактике, которая находится так далеко.
" Вовсех комментариях люди выражают желание увидеть геймплей.
Последний раз представителя вида видели профессиональные рыбаки в2009году.
Разрешение линзы позволяет видеть объекты с размером меньше длины волны света.


In [13]:
CORPUS.search_by_role(keyword_lemma='мочь')

WRONG TYPE COLLOCATE всё-таки LEMMA мочь


[['отрицание',
  'не',
  'она не может скачком\nпользователь может не справиться\nмасса не могла бы\nНикто не мог близко\nинструменты не могли быть'],
 ['самостоятельность',
  'сам',
  'пользователь может сам обвести\nпользователь сам может выбрать\nробот сам может чертить\nне может сам по\nи сам может быть'],
 ['отрицание',
  'не',
  'вовсе не может быть\nученые не смогли получить\nсна не могли начать\nоднако не может быть\nопределенно не сможет достичь'],
 ['время',
  'теперь',
  'ученые теперь могут применить\nкомпании теперь могут получить\nмы можем теперь найти\nгибрид теперь сможет отвечать'],
 ['условное наклонение',
  'бы',
  'самолет мог бы действовать\nон мог бы наблюдаться\nкоторая смогла бы разработать\nвидеоигр могли бы с\nтеоретически мог бы открыть'],
 ['косв.',
  'ли',
  'вряд ли мог претендовать\n, могут ли отклонения\n, может ли прибор\n, может ли LD\n, смогут ли такие'],
 ['узуальность',
  'не всегда',
  'магазинах не всегда могут подсказать\nкомпьютеры не всегда смо

In [14]:
CORPUS.search_by_role(keyword_lemma='видеть', role='оценка')

[['оценка', 'хорошо', 'очень хорошо видеть в\nбыл хорошо видим для']]

In [15]:
CORPUS.search_by_role(keyword_lemma='сказать', role='отрицание')

[['отрицание', 'не', 'точно сказать не могут']]